In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Load saved training and test CSV files from week 3
train = pd.read_csv("data/cleaned_train.csv")
test = pd.read_csv("data/cleaned_test.csv")

print("Training shape: ", train.shape)
print("Test shape: ", test.shape)

Training shape:  (117914, 828)
Test shape:  (12789, 828)


In [2]:
target = "ClosePrice"

city_cols = [
    col for col in train.columns
    if col.startswith("City_grouped_")
]

postal_cols = [
    col for col in train.columns
    if col.startswith("PostalCode_grouped_")
]

county_cols = [
    col for col in train.columns 
    if col.startswith("CountyOrParish_")
]

school_district_cols = [
    col for col in train.columns 
    if col.startswith("SchoolDistrict_")
]

features_sets = {
    "With Engineered Features + All Location + SchoolDistrict": [
            "LivingArea",
            "BedroomsTotal",
            "BathroomsTotalInteger",
            "LotSizeSquareFeet",
            "Missing_LotSizeSquareFeet",
            "ViewYN",
            "WaterfrontYN",
            "BasementYN",
            "PoolPrivateYN",
            "PropertyAge",
            "BedBathRatio"
        ] + city_cols + postal_cols + county_cols + school_district_cols,

    "With Everything + Missing_YearBuilt": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "Missing_YearBuilt",
        "BedBathRatio"
    ]    + city_cols + postal_cols + county_cols + school_district_cols

}


For each model tested in previous weeks, now we also compute the metrics MAPE and  MdAPE in addition to $\text{R}^2$. The number of feature sets to be tested was reduced to simplify and maintain consistency in the comparison of the model performances. The two feature sets chosen above were carried forward from week 7 as being the ones that explained the most variability in prediction of ClosePrice based on the $\text{R}^2$ values obtained in each of the models.

Baseline Linear Regression

In [3]:
linear_results = []
linear_predictions = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = LinearRegression()      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Compute MAPE
    mape_train = mean_absolute_percentage_error(y_train, y_pred_train) * 100
    mape_test = mean_absolute_percentage_error(y_test, y_pred_test) * 100

    # Compute MdAPE
    mdape_train = np.median(np.abs((y_train - y_pred_train) / y_train) * 100)
    mdape_test = np.median(np.abs((y_test - y_pred_test) / y_test) * 100)

    # Save results
    linear_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 4),
        "Test R^2": round(r2_test, 4),
        "Training MAPE": round(mape_train, 2),
        "Test MAPE": round(mape_test, 2),
        "Training MdAPE": round(mdape_train, 2),
        "Test MdAPE": round(mdape_test, 2)
    })

    linear_predictions.append(
        pd.DataFrame({
            "Model": "Linear Regression",
            "Feature Set": name,
            "Actual": y_test.values,
            "Predicted": y_pred_test
        })
    )

Decision Tree

In [4]:
# Decision Tree Regressor
decision_tree_results = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = DecisionTreeRegressor(random_state=24)      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Compute MAPE
    mape_train = mean_absolute_percentage_error(y_train, y_pred_train) * 100
    mape_test = mean_absolute_percentage_error(y_test, y_pred_test) * 100
    
    # Compute MdAPE
    mdape_train = np.median(np.abs((y_train - y_pred_train) / y_train) * 100)
    mdape_test = np.median(np.abs((y_test - y_pred_test) / y_test) * 100)

    # Save results
    decision_tree_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 4),
        "Test R^2": round(r2_test, 4),
        "Training MAPE": round(mape_train, 2),
        "Test MAPE": round(mape_test, 2),
        "Training MdAPE": round(mdape_train, 2),
        "Test MdAPE": round(mdape_test, 2)
    })


Random Forest Regressor

In [5]:
# Random Forest Regressor
random_forest_results = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = RandomForestRegressor(random_state=24, n_estimators=100, n_jobs=1)      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Compute MAPE
    mape_train = mean_absolute_percentage_error(y_train, y_pred_train) * 100
    mape_test = mean_absolute_percentage_error(y_test, y_pred_test) * 100
    
    # Compute MdAPE
    mdape_train = np.median(np.abs((y_train - y_pred_train) / y_train) * 100)
    mdape_test = np.median(np.abs((y_test - y_pred_test) / y_test) * 100)

    # Save results
    random_forest_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 4),
        "Test R^2": round(r2_test, 4),
        "Training MAPE": round(mape_train, 2),
        "Test MAPE": round(mape_test, 2),
        "Training MdAPE": round(mdape_train, 2),
        "Test MdAPE": round(mdape_test, 2)
    })


XGBoost (Tuned)

In [6]:
xgboost_results = []

for name, features in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = XGBRegressor(random_state=28,
                         n_estimators=500,
                         max_depth=8,
                         learning_rate=0.1)      # Apply the XG Boost Model with default parameter values
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Compute MAPE
    mape_train = mean_absolute_percentage_error(y_train, y_pred_train) * 100
    mape_test = mean_absolute_percentage_error(y_test, y_pred_test) * 100
    
    # Compute MdAPE
    mdape_train = np.median(np.abs((y_train - y_pred_train) / y_train) * 100)
    mdape_test = np.median(np.abs((y_test - y_pred_test) / y_test) * 100)

    # Save results
    xgboost_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 4),
        "Test R^2": round(r2_test, 4),
        "Training MAPE": round(mape_train, 2),
        "Test MAPE": round(mape_test, 2),
        "Training MdAPE": round(mdape_train, 2),
        "Test MdAPE": round(mdape_test, 2)
    })

Display results from each model

In [7]:
linear_results_df = pd.DataFrame(linear_results)
linear_results_df

,Feature Set,Number of Features,Training R^2,Test R^2,Training MAPE,Test MAPE,Training MdAPE,Test MdAPE
0,With Engineered Features + All Location + Scho...,795,0.4274,0.4408,1035.09,60.96,32.06,31.6
1,With Everything + Missing_YearBuilt,796,0.4274,0.4408,1035.09,60.96,32.06,31.6


In [8]:
decision_tree_results = pd.DataFrame(decision_tree_results)
decision_tree_results

,Feature Set,Number of Features,Training R^2,Test R^2,Training MAPE,Test MAPE,Training MdAPE,Test MdAPE
0,With Engineered Features + All Location + Scho...,795,0.9997,0.6504,0.18,31.22,0.0,12.50
1,With Everything + Missing_YearBuilt,796,0.9997,0.6524,0.18,31.45,0.0,12.37


In [9]:
random_forest_results = pd.DataFrame(random_forest_results)
random_forest_results

,Feature Set,Number of Features,Training R^2,Test R^2,Training MAPE,Test MAPE,Training MdAPE,Test MdAPE
0,With Engineered Features + All Location + Scho...,795,0.9742,0.8105,281.08,25.13,3.49,9.49
1,With Everything + Missing_YearBuilt,796,0.9742,0.8104,295.48,25.16,3.49,9.55


XGBoost Tuned

In [10]:
xgboost_results_df = pd.DataFrame(xgboost_results)
xgboost_results_df

,Feature Set,Number of Features,Training R^2,Test R^2,Training MAPE,Test MAPE,Training MdAPE,Test MdAPE
0,With Engineered Features + All Location + Scho...,795,0.9042,0.8346,739.16,25.70,10.00,10.97
1,With Everything + Missing_YearBuilt,796,0.9037,0.8356,749.86,25.74,10.08,11.12


Price Band Analysis

In [11]:
price_bands_label = ['<$500K', '$500K-$1M', '$1M-2M', '>$2M']
price_bins = [0, 500000, 1000000, 2000000, np.inf]

price_band_results = []

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=28),
    "Random Forest": RandomForestRegressor(random_state=28),
    "XGBoost": XGBRegressor(
        random_state=28,
        n_estimators=500,
        max_depth=8,
        learning_rate=0.1
    )
}

for model_name, model in models.items():

    for name, features in features_sets.items():

        X_train = train[features]
        y_train = train[target]

        X_test = test[features]
        y_test = test[target]

        # Fit model
        model.fit(X_train, y_train)

        # Predictions
        y_pred_test = model.predict(X_test)

        # Price bands
        price_bands = pd.cut(
        y_test,
        bins=price_bins,
        labels=price_bands_label,
        include_lowest=True
        )

        # Calculate metrics by band
        for band in price_bands_label:

            mask = price_bands == band

            actual = y_test[mask]
            predicted = y_pred_test[mask]

            if len(actual) < 2:
                continue

            r2 = r2_score(actual, predicted)

            mape = mean_absolute_percentage_error(actual, predicted) * 100

            mdape = np.median(np.abs((actual - predicted) / actual) * 100)

            price_band_results.append({
                "Model": model_name,
                "Feature Set": name,
                "Price Band": band,
                "N": len(actual),
                "R²": round(r2, 4),
                "MAPE": round(mape, 2),
                "MdAPE": round(mdape, 2)
            })



In [12]:
price_band_results_df = pd.DataFrame(price_band_results)
price_band_results_df

,Model,Feature Set,Price Band,N,R²,MAPE,MdAPE
0,Linear Regression,With Engineered Features + All Location + Scho...,<$500K,1818,-26.6122,202.35,83.90
1,Linear Regression,With Engineered Features + All Location + Scho...,$500K-$1M,5331,-9.1511,45.16,30.04
2,Linear Regression,With Engineered Features + All Location + Scho...,$1M-2M,3949,-2.3717,27.56,23.14
3,Linear Regression,With Engineered Features + All Location + Scho...,>$2M,1691,-0.6598,36.77,36.05
4,Linear Regression,With Everything + Missing_YearBuilt,<$500K,1818,-26.6122,202.35,83.90
5,Linear Regression,With Everything + Missing_YearBuilt,$500K-$1M,5331,-9.1511,45.16,30.04
6,Linear Regression,With Everything + Missing_YearBuilt,$1M-2M,3949,-2.3717,27.56,23.14
7,Linear Regression,With Everything + Missing_YearBuilt,>$2M,1691,-0.6598,36.77,36.05
8,Decision Tree,With Engineered Features + All Location + Scho...,<$500K,1818,-4.5759,95.46,11.84
9,Decision Tree,With Engineered Features + All Location + Scho...,$500K-$1M,5331,-4.6629,18.43,9.14


In [13]:
best_feature_sets = {
    "Linear Regression": "With Engineered Features + All Location + SchoolDistrict",
    "Decision Tree": "With Everything + Missing_YearBuilt",
    "Random Forest": "With Engineered Features + All Location + SchoolDistrict",
    "XGBoost": "With Everything + Missing_YearBuilt"
}

price_band_results_df["Feature Set"].unique()

selected_results = price_band_results_df[
    price_band_results_df.apply(
        lambda row: best_feature_sets.get(row["Model"]) == row["Feature Set"],
        axis=1
    )
]

selected_results

,Model,Feature Set,Price Band,N,R²,MAPE,MdAPE
0,Linear Regression,With Engineered Features + All Location + Scho...,<$500K,1818,-26.6122,202.35,83.90
1,Linear Regression,With Engineered Features + All Location + Scho...,$500K-$1M,5331,-9.1511,45.16,30.04
2,Linear Regression,With Engineered Features + All Location + Scho...,$1M-2M,3949,-2.3717,27.56,23.14
3,Linear Regression,With Engineered Features + All Location + Scho...,>$2M,1691,-0.6598,36.77,36.05
12,Decision Tree,With Everything + Missing_YearBuilt,<$500K,1818,-4.3404,92.89,12.00
13,Decision Tree,With Everything + Missing_YearBuilt,$500K-$1M,5331,-3.8985,18.39,9.29
14,Decision Tree,With Everything + Missing_YearBuilt,$1M-2M,3949,-2.1792,21.35,14.48
15,Decision Tree,With Everything + Missing_YearBuilt,>$2M,1691,-0.0183,27.48,22.39
16,Random Forest,With Engineered Features + All Location + Scho...,<$500K,1818,-1.9270,85.24,10.45
17,Random Forest,With Engineered Features + All Location + Scho...,$500K-$1M,5331,-1.0338,13.65,7.04


In [14]:
price_band_order = ['<$500K', '$500K-$1M', '$1M-2M', '>$2M']

selected_results["Price Band"] = pd.Categorical(
    selected_results["Price Band"],
    categories=price_band_order,
    ordered=True
)

price_band_summary = selected_results.pivot(
    index=["Price Band", "N"],
    columns="Model",
    values=["MAPE", "MdAPE"]
)

price_band_summary.columns = [
    f"{model} {metric}"
    for metric, model in price_band_summary.columns
]

price_band_summary = price_band_summary.reset_index()
price_band_summary = price_band_summary.sort_values("Price Band")
price_band_summary

,Price Band,N,Decision Tree MAPE,Linear Regression MAPE,Random Forest MAPE,XGBoost MAPE,Decision Tree MdAPE,Linear Regression MdAPE,Random Forest MdAPE,XGBoost MdAPE
0,<$500K,1818,92.89,202.35,85.24,88.17,12.00,83.90,10.45,12.69
1,$500K-$1M,5331,18.39,45.16,13.65,14.47,9.29,30.04,7.04,9.41
2,$1M-2M,3949,21.35,27.56,15.28,14.94,14.48,23.14,10.86,11.31
3,>$2M,1691,27.48,36.77,19.99,19.41,22.39,36.05,16.44,16.16


In [15]:
price_band_summary.to_csv("price_band_summary.csv", index=False)

Notes for next steps:
- Will have to go back and check the values in the dataset (training and test) for the range and unusual values for ClosePrice

Results
- Tuned XGBoost Model performed best, with highest test R^2 (0.83) and narrowest gap between test MAPE and MdAPE